# Advanced Sorting Algorithms: A Comprehensive Performance and Computational Analysis
# 

In [16]:
import random
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import statistics
from memory_profiler import memory_usage
import sys

sys.setrecursionlimit(10000)

class AdvancedSortingAlgorithms:
    @staticmethod
    def insertion_sort(arr, left=0, right=None, stability_check=False):
        """
        Insertion sort with optional stability tracking
        """
        if right is None:
            right = len(arr) - 1
        
        if stability_check:
            indexed_arr = list(enumerate(arr[left:right+1]))
            for i in range(1, len(indexed_arr)):
                key = indexed_arr[i]
                j = i - 1
                while j >= 0 and indexed_arr[j][1] > key[1]:
                    indexed_arr[j + 1] = indexed_arr[j]
                    j -= 1
                indexed_arr[j + 1] = key
            return [x[1] for x in indexed_arr]
        
        for i in range(left + 1, right + 1):
            key = arr[i]
            j = i - 1
            while j >= left and arr[j] > key:
                arr[j + 1] = arr[j]
                j -= 1
            arr[j + 1] = key
        return arr

    @staticmethod
    def quicksort_median_of_three(arr):
        """
        Iterative Quicksort with median-of-three pivot selection
        """
        def get_median_pivot(arr, low, high):
            mid = (low + high) // 2
            a, b, c = arr[low], arr[mid], arr[high]
            if a <= b <= c or c <= b <= a:
                return mid
            elif b <= a <= c or c <= a <= b:
                return low
            else:
                return high

        def partition(arr, low, high):
            # Median-of-three pivot selection
            pivot_index = get_median_pivot(arr, low, high)
            pivot = arr[pivot_index]
            
            # Swap pivot to end
            arr[pivot_index], arr[high] = arr[high], arr[pivot_index]
            
            # Partition
            store_index = low
            for i in range(low, high):
                if arr[i] < pivot:
                    arr[store_index], arr[i] = arr[i], arr[store_index]
                    store_index += 1
            
            # Move pivot to its final place
            arr[store_index], arr[high] = arr[high], arr[store_index]
            return store_index

        def iterative_quicksort(arr):
            # Using an explicit stack instead of recursion
            size = len(arr)
            stack = [(0, size - 1)]
            
            while stack:
                low, high = stack.pop()
                
                # If the segment has more than one element
                if low < high:
                    # Partition the array
                    pivot_index = partition(arr, low, high)
                    
                    # Push subarrays to stack
                    # Only push if the subarray has more than one element
                    if pivot_index - 1 > low:
                        stack.append((low, pivot_index - 1))
                    if pivot_index + 1 < high:
                        stack.append((pivot_index + 1, high))
            
            return arr

        arr_copy = arr.copy()
        return iterative_quicksort(arr_copy)

    @staticmethod
    def timsort(arr):
        """
        Timsort implementation
        """
        min_run = 32
        n = len(arr)
        
        # Sorting individual subarrays of size min_run
        for i in range(0, n, min_run):
            AdvancedSortingAlgorithms.insertion_sort(arr, i, min((i + min_run - 1), n - 1))
        
        # Merging sorted subarrays
        size = min_run
        while size < n:
            for start in range(0, n, size * 2):
                mid = start + size - 1
                end = min((start + size * 2 - 1), (n - 1))
                
                # Merging two sorted subarrays
                left = arr[start:mid + 1]
                right = arr[mid + 1:end + 1]
                
                # Merge process
                i = j = 0
                merged = []
                while i < len(left) and j < len(right):
                    if left[i] <= right[j]:
                        merged.append(left[i])
                        i += 1
                    else:
                        merged.append(right[j])
                        j += 1
                
                # Adding remaining elements
                merged.extend(left[i:])
                merged.extend(right[j:])
                
                # Updating original array
                arr[start:end + 1] = merged
            
            size *= 2
        
        return arr

    @staticmethod
    def heapsort(arr):
        """
        Heapsort implementation
        """
        def heapify(arr, n, i):
            largest = i
            left = 2 * i + 1
            right = 2 * i + 2
            
            # Checking if left child exists and is greater than root
            if left < n and arr[left] > arr[largest]:
                largest = left
            
            # Checking if right child exists and is greater than largest so far
            if right < n and arr[right] > arr[largest]:
                largest = right
            
            # Changing root if needed
            if largest != i:
                arr[i], arr[largest] = arr[largest], arr[i]
                
                # Heapify the root
                heapify(arr, n, largest)
        
        arr_copy = arr.copy()
        n = len(arr_copy)
        
        # Build max heap
        for i in range(n // 2 - 1, -1, -1):
            heapify(arr_copy, n, i)
        
        # Extracting elements from heap one by one
        for i in range(n - 1, 0, -1):
            arr_copy[0], arr_copy[i] = arr_copy[i], arr_copy[0]
            
            # Heapify the reduced heap
            heapify(arr_copy, i, 0)
        
        return arr_copy

    @staticmethod
    def introsort(arr):
        """
        Introsort implementation with iterative approach
        """
        def partition(arr, low, high):
            pivot = arr[high]
            i = low - 1
            
            for j in range(low, high):
                if arr[j] <= pivot:
                    i += 1
                    arr[i], arr[j] = arr[j], arr[i]
            
            arr[i + 1], arr[high] = arr[high], arr[i + 1]
            return i + 1

        def iterative_introsort(arr, max_depth):
            # Using an explicit stack instead of recursion
            stack = [(0, len(arr) - 1, max_depth)]
            
            while stack:
                low, high, depth = stack.pop()
                
                # Using insertion sort for small arrays
                if high - low + 1 <= 16:
                    AdvancedSortingAlgorithms.insertion_sort(arr, low, high)
                    continue
                
                # Switching to heapsort if recursion depth is too high
                if depth == 0:
                    AdvancedSortingAlgorithms.heapsort(arr[low:high+1])
                    continue
                
                # Partitioning
                if low < high:
                    pivot_index = partition(arr, low, high)
                    
                    # Push subarrays to stack
                    if pivot_index - 1 > low:
                        stack.append((low, pivot_index - 1, depth - 1))
                    if pivot_index + 1 < high:
                        stack.append((pivot_index + 1, high, depth - 1))
        
        arr_copy = arr.copy()
        if not arr_copy:
            return arr_copy
        
        max_depth = 2 * math.floor(math.log2(len(arr_copy)))
        iterative_introsort(arr_copy, max_depth)
        return arr_copy

class SortingBenchmark:
    @staticmethod
    def generate_test_array(size, distribution='random'):
        np.random.seed(42)
        if distribution == 'random':
            return list(np.random.randint(0, 10000, size))
        elif distribution == 'sorted':
            return list(np.sort(np.random.randint(0, 10000, size)))
        elif distribution == 'reverse_sorted':
            return list(np.sort(np.random.randint(0, 10000, size))[::-1])
        elif distribution == 'partially_sorted':
            arr = list(np.sort(np.random.randint(0, 10000, size)))
            for _ in range(size // 10):
                idx1, idx2 = np.random.choice(size, 2, replace=False)
                arr[idx1], arr[idx2] = arr[idx2], arr[idx1]
            return arr
        elif distribution == 'uniform':
            return list(np.random.uniform(0, 1, size))
        elif distribution == 'gaussian':
            return list(np.random.normal(0, 1, size))
        elif distribution == 'duplicates':
            return list(np.random.choice([0, 1, 2, 3], size))

    @staticmethod
    def benchmark_sorting_algorithms(sizes, distributions, algorithms):
        results = {algo.__name__: {dist: [] for dist in distributions} for algo in algorithms}
        memory_results = {algo.__name__: {dist: [] for dist in distributions} for algo in algorithms}
        
        for size in sizes:
            for distribution in distributions:
                test_array = SortingBenchmark.generate_test_array(size, distribution)
                for algo in algorithms:
                    arr_copy = test_array.copy()
                    
                    # Measure memory usage with a try-except to handle potential errors
                    try:
                        mem_usage = memory_usage((algo, (arr_copy,)), interval=0.1, max_iterations=100)
                    except Exception as e:
                        print(f"Memory profiling error for {algo.__name__}: {e}")
                        mem_usage = [0]
                    
                    start_time = time.time()
                    algo(arr_copy)
                    end_time = time.time()
                    
                    results[algo.__name__][distribution].append(end_time - start_time)
                    memory_results[algo.__name__][distribution].append(max(mem_usage))
        
        return results, memory_results

    @staticmethod
    def check_stability(arr, sorted_arr):
        """
        Check if sorting preserves relative order of equal elements
        """
        def count_inversions(original, sorted_arr):
            # Counting number of elements that change relative order
            return sum(1 for x, y in zip(original, sorted_arr) if x != y)
        
        return count_inversions(arr, sorted_arr)

    @staticmethod
    def complexity_analysis(sizes, algorithm):
        """
        Compare theoretical vs empirical time complexity
        """
        theoretical_complexity = {
            'timsort': 'O(n log n)',
            'quicksort_median_of_three': 'O(n log n)',
            'heapsort': 'O(n log n)',
            'introsort': 'O(n log n)'
        }
        
        empirical_results = []
        for size in sizes:
            arr = SortingBenchmark.generate_test_array(size)
            start_time = time.time()
            algorithm(arr)
            end_time = time.time()
            empirical_results.append(end_time - start_time)
        
        return {
            'theoretical': theoretical_complexity[algorithm.__name__],
            'empirical_times': empirical_results
        }

    @staticmethod
    def plot_performance(results, memory_results, sizes, distributions):
        import os
        
        os.makedirs('sorting_plots', exist_ok=True)
        
        for dist in distributions:
            # Time Performance Plot
            plt.figure(figsize=(12, 8))
            plt.title(f"Sorting Performance: {dist.capitalize()} Distribution")
            plt.xlabel("Array Size")
            plt.ylabel("Execution Time (s)")
            for algo, timings in results.items():
                plt.plot(sizes, timings[dist], label=algo)
            plt.legend()
            plt.grid()
            
            # Save time performance plot
            time_plot_filename = os.path.join('sorting_plots', f'{dist}_performance.png')
            plt.savefig(time_plot_filename)
            plt.close()  # Close the plot to free up memory
            
            # Memory Usage Plot
            plt.figure(figsize=(12, 8))
            plt.title(f"Memory Usage: {dist.capitalize()} Distribution")
            plt.xlabel("Array Size")
            plt.ylabel("Memory Usage (MB)")
            for algo, mem in memory_results.items():
                plt.plot(sizes, mem[dist], label=algo)
            plt.legend()
            plt.grid()
            
            # Save memory usage plot
            memory_plot_filename = os.path.join('sorting_plots', f'{dist}_memory_usage.png')
            plt.savefig(memory_plot_filename)
            plt.close()  # Close the plot to free up memory
            
            print(f"Plots for {dist} distribution saved:")
            print(f"- Performance plot: {time_plot_filename}")
            print(f"- Memory usage plot: {memory_plot_filename}")

def main():
    sizes = [100, 500, 1000, 5000, 10000]  # Reduced sizes to prevent potential memory issues
    distributions = ['random', 'sorted', 'reverse_sorted', 
                     'partially_sorted', 'duplicates', 'gaussian']
    algorithms = [
        AdvancedSortingAlgorithms.timsort,
        AdvancedSortingAlgorithms.quicksort_median_of_three,
        AdvancedSortingAlgorithms.heapsort,
        AdvancedSortingAlgorithms.introsort
    ]

    print("Comprehensive Sorting Algorithm Analysis")
    
    results, memory_results = SortingBenchmark.benchmark_sorting_algorithms(
        sizes, distributions, algorithms)
    
    SortingBenchmark.plot_performance(results, memory_results, sizes, distributions)
    
    print("\nStability Analysis:")
    for dist in distributions:
        print(f"\nDistribution: {dist}")
        test_arr = SortingBenchmark.generate_test_array(1000, dist)
        for algo in algorithms:
            sorted_arr = algo(test_arr.copy())
            stability_score = SortingBenchmark.check_stability(test_arr, sorted_arr)
            print(f"{algo.__name__}: Stability Score = {stability_score}")
    
    print("\nComplexity Analysis:")
    for algo in algorithms:
        complexity = SortingBenchmark.complexity_analysis(sizes, algo)
        print(f"\n{algo.__name__}:")
        print(f"Theoretical Complexity: {complexity['theoretical']}")
        print("Empirical Time Scaling:", 
              [f"{size}: {time:.4f}s" for size, time in zip(sizes, complexity['empirical_times'])])

if __name__ == "__main__":
    main()

Comprehensive Sorting Algorithm Analysis
Plots for random distribution saved:
- Performance plot: sorting_plots/random_performance.png
- Memory usage plot: sorting_plots/random_memory_usage.png
Plots for sorted distribution saved:
- Performance plot: sorting_plots/sorted_performance.png
- Memory usage plot: sorting_plots/sorted_memory_usage.png
Plots for reverse_sorted distribution saved:
- Performance plot: sorting_plots/reverse_sorted_performance.png
- Memory usage plot: sorting_plots/reverse_sorted_memory_usage.png
Plots for partially_sorted distribution saved:
- Performance plot: sorting_plots/partially_sorted_performance.png
- Memory usage plot: sorting_plots/partially_sorted_memory_usage.png
Plots for duplicates distribution saved:
- Performance plot: sorting_plots/duplicates_performance.png
- Memory usage plot: sorting_plots/duplicates_memory_usage.png
Plots for gaussian distribution saved:
- Performance plot: sorting_plots/gaussian_performance.png
- Memory usage plot: sorting_p